# Project Canary — Harvest-Recovery Forecast

**Purpose:** reproduce the complete forecasting workflow in a form the capstone team can run and defend.

**Business question:** using only records known on a review date, how much additional population loss should we expect, and what final recovery does that imply against the 95% goal?

This notebook does not calculate the independent rules-based risk score and does not prove that any input causes recovery to change.

## 1. Define Y, X, and the unit of analysis

- **Y target:** additional population loss after the review date = current percentage alive − completed-cycle recovery proxy.
- **Final output:** current percentage alive − predicted additional loss.
- **Important:** this is the agreed capstone recovery proxy, not a verified harvest-event label.
- **One outcome:** one building in one completed cycle.
- **One training snapshot:** that building's facts known at a selected age. To avoid overweighting long cycles, training retains Days 7, 14, 21, 28, plus the last eligible pre-outcome snapshot.
- **Candidate X inputs:** production age; current survival; recent mortality; weight gap and measurement freshness; and temperature/humidity deviations from approved age bands. Feed is withheld until its recorded unit is confirmed. The compact set excludes building identity, raw inventory size, and algebraic duplicates.

In [1]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd

ROOT = Path.cwd()
if not (ROOT / "canary").exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
DATA_PATH = ROOT / "data" / "FARM HARVEST DATA.xlsx"
MODEL_READY_DIR = ROOT / "outputs" / "model_ready"

from canary import load_workbook

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 140)
dataset = load_workbook(DATA_PATH)
print(f"Source: {dataset.source_name}")
print(f"Canonical building-day rows: {len(dataset.daily):,}")
print(f"Recorded building-cycles: {len(dataset.cycles):,}")
print(f"Blocking data-quality checks passed: {dataset.quality.passed}")
print(f"Non-blocking warnings: {len(dataset.quality.warnings)}")

Source: FARM HARVEST DATA.xlsx
Canonical building-day rows: 1,624
Recorded building-cycles: 34
Blocking data-quality checks passed: True
Non-blocking warnings: 2


In [2]:
from canary import build_modeling_snapshots, build_recovery_training_snapshots, train_outcome_model

daily_snapshots = build_modeling_snapshots(dataset, "recovery")
training_snapshots = build_recovery_training_snapshots(dataset)
coverage = pd.DataFrame({
    "Measure": ["Complete cycles", "Distinct building outcomes", "All leakage-safe daily snapshots", "Balanced decision snapshots"],
    "Count": [training_snapshots["cycle_id"].nunique(), training_snapshots[["cycle_id", "building_id"]].drop_duplicates().shape[0], len(daily_snapshots), len(training_snapshots)],
})
coverage

In [3]:
exported = pd.read_csv(MODEL_READY_DIR / "recovery_training.csv")
assert len(exported) == len(training_snapshots)
expected_keys = set(zip(training_snapshots.cycle_id.astype(str), training_snapshots.building_id, training_snapshots.as_of_date.astype(str)))
exported_keys = set(zip(exported.cycle_id.astype(str), exported.building_id, exported.as_of_date.astype(str)))
assert exported_keys == expected_keys
print(f"Export reconciliation passed: the CSV contains the exact {len(exported)} balanced recovery snapshots.")

Export reconciliation passed: the CSV contains the exact 151 balanced recovery snapshots.


## 2. Preprocessing and validation

1. Convert the workbook to one canonical building-day row; zone rows are aggregated before modeling.
2. Construct every snapshot with records dated on or before its review date; later records are excluded.
3. Give each building-cycle equal total weight despite repeated checkpoints.
4. Use **nested leave-one-complete-cycle-out cross-validation**: the outer loop tests a completely unseen cycle; the inner loop tunes only within the remaining cycles. Imputation and scaling stay inside those folds.
5. Compare exactly five candidates: age-band remaining-loss baseline, linear regression, Ridge regression, constrained Gradient Boosting, and constrained Extra Trees.
6. A learned recovery model must beat the baseline by at least 10% in cycle-macro MAE, keep positive whole-cycle R², and remain stable. A separate balanced target-side gate controls whether Canary may describe it as a 95% hit/miss classifier.

No random row split is used because rows from the same flock history are related and would leak information across train and test sets.

In [4]:
result = train_outcome_model(dataset, "recovery")
manifest = result.manifest
print("Operational recovery method:", manifest["selected_model"])
print("Best learned challenger:", manifest["research_champion"])
print("Champion gates:", manifest["champion_gates"])
print("Model version:", manifest["model_version"])
print("Selected X inputs:")
for feature in manifest["feature_columns"]:
    print(" -", feature)

Operational recovery method: remaining_loss_linear
Best learned challenger: remaining_loss_linear
Champion gates: {'baseline': 'age_band_remaining_loss', 'baseline_improvement_pct': 16.067318162088867, 'requires_at_least_10pct_mae_improvement': True, 'requires_positive_r2': True, 'requires_stable_worst_cycle': True, 'regression_gate_passed': True, 'requires_better_than_majority_target_side_accuracy': True, 'requires_recall_for_both_target_sides': True, 'requires_at_least_60pct_balanced_target_accuracy': True, 'target_classification_gate_passed': True, 'classification_claim_allowed': True, 'operational_fallback_applied': False}
Model version: recovery-3.2.0
Selected X inputs:
 - cycle_day
 - percentage_alive
 - population_loss_pct
 - mortality_recent_3d_per_1000
 - mortality_trend_delta_per_1000
 - weight_gap_pct
 - weight_staleness_days
 - temperature_deviation_from_band_c
 - humidity_deviation_from_band_pp
 - environment_out_of_band_days_7d
 - environment_staleness_days
 - is_lags_bui

## 3. Candidate comparison

In [5]:
comparison = pd.DataFrame([
    {
        "Candidate": entry["model"],
        "Available": entry["available"],
        "Role": "Operational" if entry["model"] == manifest["selected_model"] else "Best learned challenger" if entry["model"] == manifest["research_champion"] else "Compared",
        "MAE (points)": manifest["metrics"].get(entry["model"], {}).get("mae", np.nan) * 100,
        "Cycle-macro MAE (points)": manifest["metrics"].get(entry["model"], {}).get("cycle_macro_mae", np.nan) * 100,
        "RMSE (points)": manifest["metrics"].get(entry["model"], {}).get("rmse", np.nan) * 100,
        "R²": manifest["metrics"].get(entry["model"], {}).get("r2", np.nan),
        "Target-side accuracy": manifest["metrics"].get(entry["model"], {}).get("target_side_accuracy", np.nan),
    }
    for entry in manifest["candidate_registry"]
]).sort_values("Cycle-macro MAE (points)")
comparison.round({"MAE (points)": 2, "Cycle-macro MAE (points)": 2, "RMSE (points)": 2, "R²": 3, "Bias (points)": 2, "Target-side accuracy": 3})

In [6]:
cycle_performance = pd.DataFrame.from_dict(manifest["selected_metrics"]["cycle"], orient="index")
cycle_performance.index.name = "Held-out cycle"
cycle_performance.assign(
    mae_points=cycle_performance.mae * 100,
    rmse_points=cycle_performance.rmse * 100,
    bias_points=cycle_performance.bias * 100,
)[["rows", "mae_points", "rmse_points", "bias_points"]].round(2)

In [7]:
selected = manifest["selected_metrics"]
print(f"Selected held-out MAE: {selected['mae']*100:.2f} percentage points")
print(f"Selected held-out RMSE: {selected['rmse']*100:.2f} percentage points")
print(f"80% empirical error half-width: ±{selected['uncertainty_half_width_80']*100:.2f} points")
print(f"Target-side accuracy: {selected['target_side_accuracy']:.1%}")
print(f"Majority baseline accuracy: {selected['majority_side_accuracy']:.1%}")

Selected held-out MAE: 1.74 percentage points
Selected held-out RMSE: 2.57 percentage points
80% empirical error half-width: ±2.83 points
Target-side accuracy: 90.1%
Majority baseline accuracy: 87.4%


**Interpretation:** the operational method is read directly from the versioned manifest. Selection prioritizes cycle-balanced MAE, positive R², stability across held-out cycles, and simplicity. The current release uses ordinary linear remaining-loss regression because it improves cycle-balanced MAE over the age-band baseline and is as accurate as Ridge while remaining easier to explain. Its R² is still low and its at/above-95% recall is weak, so present it as an experimental continuous estimate with uncertainty—not a probability or guarantee of target attainment.

## 4. What the selected model relies on

In [8]:
importance = pd.DataFrame(manifest["held_out_permutation_importance"])
importance.head(10).rename(columns={
    "feature": "Input",
    "mean_mae_increase": "Held-out MAE increase",
    "relative_importance_pct": "Relative held-out reliance (%)",
}).round(4)

These are out-of-fold permutation importances from complete unseen cycles. They show predictive reliance and are **associations, not causal effects**.

## 4B. Held-out SHAP — direction and magnitude

In [9]:
shap_summary = pd.DataFrame(manifest["held_out_shap_importance"])
shap_summary.head(10).rename(columns={
    "feature": "Input",
    "mean_abs_shap_recovery": "Mean absolute SHAP effect",
    "relative_mean_abs_shap_pct": "Relative SHAP reliance (%)",
    "direction_when_value_increases": "General direction when higher",
})[["Input", "Mean absolute SHAP effect", "Relative SHAP reliance (%)", "General direction when higher"]].round(4)

SHAP was calculated on each complete outer held-out cycle for the strongest tree challenger—not on its training fit. It is shown as a non-linear sensitivity analysis and does **not** explain the operational linear model. Because the tree predicts **additional loss**, SHAP signs are negated so positive values mean the feature raised final recovery and negative values mean it lowered final recovery. This explains model behavior; it does not prove that intervening on the feature will cause the predicted change.

## 5. Day 14 held-out proof and one complete example

In [10]:
def cycle_bootstrap_mae(frame, error_column, repeats=5000, seed=42):
    # Bootstrap whole cycles, never individual rows, to preserve grouped evidence.
    rng = np.random.default_rng(seed)
    grouped = {cycle: group for cycle, group in frame.groupby("cycle_id")}
    cycles = np.array(list(grouped))
    estimates = []
    for _ in range(repeats):
        selected = rng.choice(cycles, size=len(cycles), replace=True)
        errors = np.concatenate([grouped[cycle][error_column].to_numpy(float) for cycle in selected])
        estimates.append(np.mean(np.abs(errors)))
    return np.quantile(estimates, [0.025, 0.975])

In [11]:
day14 = pd.DataFrame(manifest["day14_backtest"])
day14["error_points"] = day14["error"] * 100
day14["absolute_error_points"] = day14["absolute_error"] * 100
ci = cycle_bootstrap_mae(day14, "error_points")
metrics = manifest["day14_backtest_metrics"]
print(f"Day 14 building outcomes: {metrics['building_cycles']}")
print(f"Day 14 MAE: {metrics['mae']*100:.2f} points")
print(f"Cycle-bootstrap 95% interval for Day 14 MAE: {ci[0]:.2f} to {ci[1]:.2f} points")
example = day14.iloc[0]
print("\nExample")
print(f"Cycle/building: {example.cycle_id} / {example.building_id}")
print(f"Day 14 held-out projection: {example.predicted:.1%}")
print(f"Last-recorded actual proxy: {example.actual:.1%}")
print(f"Error = projected - actual: {example.error_points:+.2f} percentage points")
day14.head(8)[["cycle_id", "building_id", "predicted", "actual", "error_points"]]

Day 14 building outcomes: 31
Day 14 MAE: 1.95 points
Cycle-bootstrap 95% interval for Day 14 MAE: 1.28 to 2.74 points

Example
Cycle/building: 2025-2 / Tags 1
Day 14 held-out projection: 92.7%
Last-recorded actual proxy: 94.3%
Error = projected - actual: -1.53 percentage points


## 6. Why SMOTE or oversampling is not used

- The outcome is continuous regression, while standard SMOTE is designed for classification.
- The scarce item is the number of independent building-cycle outcomes—not the number of spreadsheet rows. Synthetic rows do not create new farms or cycles.
- Interpolating flock records could create biologically implausible combinations and falsely narrow validation error.
- Oversampling before grouped validation could leak the held-out cycle.

**Safer small-data strategy used here:** simple regularized candidates, complete-cycle holdouts, balanced checkpoints, empirical uncertainty, cycle-level bootstrap intervals, and transparent limitations. The strongest improvement is collecting more standardized completed cycles with verified harvest events.

## 7. Defense takeaway

Canary's recovery output is a **nested whole-cycle-validated estimate of the agreed last-recorded recovery proxy**. The refreshed model uses 31 independent outcomes across six completed cycles. Its held-out error is roughly 1–2 percentage points, but its low R² and weak recall of the small number of outcomes at or above 95% require cautious use. Use it to size likely gaps and guide attention, not to claim certainty.